#Start of work in 21/03/26

In [ ]:
import os
import pandas as pd
import numpy as np

# Set the path to the data directory within the project
data_dir = os.path.join(os.getcwd(), 'Data')
file_name = 'KP02-Pain maps and VAS 28Jul2022.xlsx'
full_file_path = os.path.join(data_dir, file_name)

# Load the sheets
try:
    df_labels = pd.read_excel(full_file_path, sheet_name='Pain Map Labels', header=0)
    df_vas = pd.read_excel(full_file_path, sheet_name='VAS', header=[0, 1])
except Exception as e:
    print(f"Error loading Excel file: {e}")
    raise

# Flatten the two rows of headers into a single row
new_columns = []
for col in df_vas.columns:
    top_level = str(col[0])
    bottom_level = str(col[1])
    if top_level.startswith("Unnamed"):
        new_columns.append(bottom_level)
    else:
        new_columns.append(f"{top_level}_{bottom_level}")

df_vas.columns = new_columns

print("Data loaded and headers flattened!")

In [ ]:
# Fix the Hebrew 'PN' column in the VAS sheet FIRST
for col in df_vas.columns:
    if 'מתנדב' in str(col):
        df_vas.rename(columns={col: 'PN'}, inplace=True)
        break

# Fix the 'VISIT' column name in the Labels sheet
if 'VISIT' in df_labels.columns:
    df_labels.rename(columns={'VISIT': 'Visit'}, inplace=True)

# Clean and Standardize Patient IDs (PN) in both datasets
df_vas['PN'] = df_vas['PN'].astype(str).str.strip().str.upper()
df_vas['PN'] = df_vas['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

df_labels['PN'] = df_labels['PN'].astype(str).str.strip().str.upper()
df_labels['PN'] = df_labels['PN'].str.replace(r'-(\d)$', r'-0\1', regex=True)

print("Patient IDs and column names standardized! 'PN' is ready to use.")

In [ ]:
visit_mapping = {
    1: 'Baseline',
    2: 'End of treatment',
    3: 'End of trial'
}

vas_long_pieces = []
for visit_num, prefix in visit_mapping.items():
    stage_cols = [col for col in df_vas.columns if col.startswith(prefix)]
    cols_to_keep = ['PN'] + stage_cols

    df_subset = df_vas[cols_to_keep].copy()
    df_subset.columns = ['PN'] + [col.replace(f"{prefix}_", "") for col in stage_cols]

    df_subset['Visit'] = visit_num
    vas_long_pieces.append(df_subset)

df_vas_long = pd.concat(vas_long_pieces, ignore_index=True)

print("VAS data successfully reshaped! 'df_vas_long' is now ready.")

In [ ]:
# Perform the LEFT Join (Only keeps patients found in df_labels)
df_merged = pd.merge(df_labels, df_vas_long, on=['PN', 'Visit'], how='left')

# Identify the side
is_right = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Right'
is_left = df_merged['SIDE_'].astype(str).str.strip().str.title() == 'Left'

# Create the new unified columns
df_merged.loc[is_right, 'Worst'] = df_merged.loc[is_right, 'R-Worst']
df_merged.loc[is_right, 'Average'] = df_merged.loc[is_right, 'R-Average']

df_merged.loc[is_left, 'Worst'] = df_merged.loc[is_left, 'L-Worst']
df_merged.loc[is_left, 'Average'] = df_merged.loc[is_left, 'L-Average']

# Drop the original Left and Right specific columns
cols_to_drop = ['R-Worst', 'L-Worst', 'R-Average', 'L-Average']
df_merged.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Left merge and side-consolidation complete!")

In [ ]:
pd.set_option('display.max_columns', None)

total_patients = df_merged['PN'].nunique()
print(f"Final Dataset Shape: {df_merged.shape}")
print(f"Total Unique Patients from Labels: {total_patients}")
print("-" * 40)

# Display the list of unique patients to check for duplicates
print("Unique Patient IDs:")
print(sorted(df_merged['PN'].unique().tolist()))

# Display the key columns to verify
display_cols = ['PN', 'Visit', 'SIDE_', 'Worst', 'Average']
display(df_merged[display_cols].head(10))

In [ ]:
df_merged.dtypes

In [ ]:
# List of columns to remove (PN is NOT in this list)
cols_to_remove = [
    'PC', 'INCLUDED', 'GROUP', 'DOCTOR', 'Docter',
    'QUALITY', 'תאריך', 'Baseline_תאריך'
]

# This looks for any of the above names in your dataframe columns
existing_cols_to_drop = [c for c in df_merged.columns if c in cols_to_remove or any(x.upper() == c.upper() for x in cols_to_remove)]

# Drop the columns
df_merged.drop(columns=existing_cols_to_drop, inplace=True, errors='ignore')

print(f"Cleanup complete! 'PN' has been preserved.")
print(f"Remaining columns: {df_merged.columns.tolist()}")
print("-" * 40)

# Display to verify PN is still there
display(df_merged.head())

In [ ]:
# Create a single binary column: 1 for Right, 0 for Left
if 'SIDE_' in df_merged.columns:
    # Convert 'Right' to 1 and 'Left' to 0
    df_merged['side_right'] = df_merged['SIDE_'].astype(str).str.strip().str.title().map({
        'Right': 1,
        'Left': 0
    })

    # Fill any missing values with 0 and ensure integer type
    df_merged['side_right'] = df_merged['side_right'].fillna(0).astype(int)

    # Drop the original text column
    df_merged.drop(columns=['SIDE_'], inplace=True)

    print("Success: 'SIDE_' text removed. Created 'side_right' binary column (1=Right, 0=Left).")
else:
    print("Check: 'SIDE_' column not found. It may have already been processed.")

# Display to verify
display(df_merged[['PN', 'side_right', 'Worst', 'Average']].head(10))

In [ ]:
# List of columns to drop (keeping the new One-Hot columns)
cols_to_drop = ['SIDE_']

# Drop the column and verify
df_merged.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Cleanup complete! The text-based 'SIDE_' column has been removed.")
print("-" * 40)

# Display the final structure with the One-Hot columns
display(df_merged.head())

In [ ]:
# Convert integer codes (1 and 2) to Binary (1 and 0)
# Standard convention: 1 = Male, 2 = Female

if 'SEX' in df_merged.columns:
    # Map 1 to 1 (Male) and 2 to 0 (Female)
    df_merged['male'] = df_merged['SEX'].map({1: 0, 2: 1})

    # Fill any missing values with 0 and ensure integer type
    df_merged['male'] = df_merged['male'].fillna(0).astype(int)

    # Drop the original 'SEX' column
    df_merged.drop(columns=['SEX'], inplace=True)

    print("Success: Converted 'SEX' (1/2) into 'male' (1/0).")
    display(df_merged[['PN', 'male']].head(10))
else:
    print("Check: 'SEX' column not found. It may have already been renamed or dropped.")

In [ ]:
# Show the first 10 rows to verify the final mapped columns
# We are now looking at our clean, ANN-ready binary columns!
display_cols = ['PN', 'Visit', 'side_right', 'male', 'Worst', 'Average']
display(df_merged[display_cols].head(10))

In [ ]:
# Drop the original text-based SEX column
df_merged.drop(columns=['SEX'], inplace=True, errors='ignore')

print("Cleanup complete! The original 'SEX' column has been removed.")
print("-" * 40)

# Display the final structure to see your new SEX columns
display(df_merged.head())

In [ ]:
# Rename selected CSV columns before image processing
df_merged = df_merged.rename(columns={
    "Anterior_3": "Ant_3",
    "Anterior_A5": "Ant_5",
    "Anterior_A6": "Ant_6",
    "Anterior_A7": "Ant_7",
    "Anterior_A9": "Ant_9"
})

In [ ]:
# Create unified Lateral column as OR of Lateral_LJL and Lateral_LCL
if 'Lateral_LJL' in df_merged.columns and 'Lateral_LCL' in df_merged.columns:
    df_merged['Lateral'] = (
        df_merged['Lateral_LJL'].fillna(0).astype(int) |
        df_merged['Lateral_LCL'].fillna(0).astype(int)
    )

# Create unified Medial column as OR of Medial_MJL and Medial_MCL
if 'Medial_MJL' in df_merged.columns and 'Medial_MCL' in df_merged.columns:
    df_merged['Medial'] = (
        df_merged['Medial_MJL'].fillna(0).astype(int) |
        df_merged['Medial_MCL'].fillna(0).astype(int)
    )

# Drop the original split columns after creating the unified ones
df_merged.drop(
    columns=['Lateral_LJL', 'Lateral_LCL', 'Medial_MJL', 'Medial_MCL'],
    inplace=True,
    errors='ignore'
)

print("Created unified 'Lateral' and 'Medial' columns and removed the original split columns.")
display(df_merged.head())

In [ ]:
# Create pain_label using the same values as PF_DIAGNOSIS
if 'PF_DIAGNOSIS' not in df_merged.columns:
    raise KeyError("df_merged must contain 'PF_DIAGNOSIS' to create pain_label.")

# Copy PF_DIAGNOSIS directly into pain_label
# Coerce to numeric to keep a clean label column when possible.
df_merged['pain_label'] = pd.to_numeric(df_merged['PF_DIAGNOSIS'], errors='coerce')

print("Created 'pain_label' from 'PF_DIAGNOSIS'.")
print("Value counts (including NaN):")
print(df_merged['pain_label'].value_counts(dropna=False).sort_index())

display(df_merged[['PN', 'Visit', 'PF_DIAGNOSIS', 'pain_label']].head(10))

In [ ]:
# Make sure the needed anterior columns are numeric
ant_cols = ["Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9"]
for col in ant_cols:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce").fillna(0).astype(int)

# Create Pat column:
# Pat = 1 if at least 2 of Ant_5, Ant_6, Ant_7 are 1
if all(col in df_merged.columns for col in ["Ant_5", "Ant_6", "Ant_7"]):
    df_merged["Pat"] = (
        df_merged[["Ant_5", "Ant_6", "Ant_7"]].sum(axis=1) >= 2
    ).astype(int)

# Create Ext column:
# Ext = 1 if at least 2 of Ant_3, Ant_6, Ant_9 are 1
if all(col in df_merged.columns for col in ["Ant_3", "Ant_6", "Ant_9"]):
    df_merged["Ext"] = (
        df_merged[["Ant_3", "Ant_6", "Ant_9"]].sum(axis=1) >= 2
    ).astype(int)

print("Pat and Ext columns were added to df_merged.")
display(df_merged[["PN", "Visit", "side_right", "Ant_3", "Ant_5", "Ant_6", "Ant_7", "Ant_9", "Pat", "Ext"]].head())

In [ ]:
# Reorder columns so that Lateral, Medial, Pat, Ext come right after Ant_9
priority_cols = ["Lateral", "Medial", "Pat", "Ext"]

if "Ant_9" in df_merged.columns:
    cols = list(df_merged.columns)

    # Remove the four columns from their current positions
    cols = [col for col in cols if col not in priority_cols]

    # Find where Ant_9 is
    ant9_index = cols.index("Ant_9")

    # Insert the four columns right after Ant_9
    new_cols = cols[:ant9_index + 1] + priority_cols + cols[ant9_index + 1:]

    # Keep only columns that actually exist
    new_cols = [col for col in new_cols if col in df_merged.columns]

    df_merged = df_merged[new_cols]

print("Columns reordered successfully.")
display(df_merged.head())

#End of try of 21/03/2026


# Image EDA

This section explores the image dataset before preprocessing.
The goal is to understand image formats, dimensions, aspect ratios, color modes, and possible quality issues such as corrupted or very small images.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

# Define the base images directory
base_dir = Path('Images')

# List all subject/session directories (ignore files and system folders)
subject_dirs = [d for d in base_dir.iterdir() if d.is_dir() and not d.name.startswith('.') and d.name != 'Icon']

# Define subdirectory names for each region
region_names = {
    'Ant': 'Ant',
    'Post': 'Post',
    'Med': ['L/Med', 'R/Med'],
    'Lat': ['L/Let', 'R/Let']
}

image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']

def collect_image_info(paths):
    records = []
    corrupted_files = []
    if not isinstance(paths, list):
        paths = [paths]
    for path in paths:
        # Recursively search for images in all subfolders
        for ext in image_extensions:
            for img_path in Path(path).rglob(f'*{ext}'):
                try:
                    with Image.open(img_path) as img:
                        width, height = img.size
                        mode = img.mode
                        # Find the subject and region by traversing parents
                        parents = list(img_path.parents)
                        subject = ''
                        region = ''
                        for p in parents:
                            if p.parent == base_dir:
                                subject = p.name
                            if p.name in ['Ant', 'Post', 'Med', 'Let', 'Lat']:
                                region = 'Lat' if p.name == 'Let' else p.name
                        records.append({
                            'file_name': img_path.name,
                            'file_path': str(img_path),
                            'subject': subject,
                            'region': region,
                            'extension': img_path.suffix.lower(),
                            'width': width,
                            'height': height,
                            'aspect_ratio': width / height if height != 0 else np.nan,
                            'mode': mode
                        })
                except Exception as e:
                    corrupted_files.append({
                        'file_name': img_path.name,
                        'file_path': str(img_path),
                        'error': str(e)
                    })
    return pd.DataFrame(records), pd.DataFrame(corrupted_files)

# Collect data for each region across all subjects, recursively
dfs = {}
corrupted_dfs = {}
for region, subdirs in region_names.items():
    all_paths = []
    if isinstance(subdirs, str):
        for subj in subject_dirs:
            # Recursively search for region-named folders anywhere under the subject
            for region_path in subj.rglob(subdirs):
                if region_path.exists():
                    all_paths.append(region_path)
    else:
        for subj in subject_dirs:
            for subdir in subdirs:
                for region_path in subj.rglob(subdir):
                    if region_path.exists():
                        all_paths.append(region_path)
    dfs[region], corrupted_dfs[region] = collect_image_info(all_paths)

print(f"Ant images: {len(dfs['Ant'])}")
print(f"Post images: {len(dfs['Post'])}")
print(f"Med images: {len(dfs['Med'])}")
print(f"Lat images: {len(dfs['Lat'])}")

In [ ]:
# Present the DataFrame of Post images
display(dfs['Post'])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

# Define the base images directory and valid extensions
base_dir = Path('Images')
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

In [ ]:
records = []
corrupted_files = []

# Iterate through every file in the base_dir and its subdirectories exactly ONCE
for img_path in base_dir.rglob('*'):
    
    # Check if it's a file and has a valid image extension
    if img_path.is_file() and img_path.suffix.lower() in image_extensions:
        
        # Ignore system files and the 'Icon' folder (case-insensitive check)
        if any(part.startswith('.') or part.lower() == 'icon' for part in img_path.parts):
            continue
            
        try:
            subject = None
            side = None
            region = None
            
            # Traverse all parts in the path to find region and side, ignoring case and spaces
            for i, part in enumerate(img_path.parts):
                
                clean_part = part.strip().lower() # Normalize the string to catch weird formatting
                
                # Subject: first folder under Images (keep original casing for the subject ID)
                if i > 0 and img_path.parts[i-1] == base_dir.name and subject is None:
                    subject = part
                    
                # Side: any folder named L or R
                if clean_part in ['l', 'r'] and side is None:
                    side = clean_part.upper()
                    
                # Region: any folder named Lat, Med, Let, Ant, Post (deepest match wins)
                if clean_part in ['lat', 'med', 'let', 'ant', 'post']:
                    region = 'Lat' if clean_part == 'let' else clean_part.capitalize()
                    
            with Image.open(img_path) as img:
                width, height = img.size
                mode = img.mode
                
            records.append({
                'file_name': img_path.name,
                'file_path': str(img_path),
                'subject': subject,
                'side': side,
                'region': region,
                'extension': img_path.suffix.lower(),
                'width': width,
                'height': height,
                'aspect_ratio': width / height if height != 0 else np.nan,
                'mode': mode
            })
            
        except Exception as e:
            corrupted_files.append({
                'file_name': img_path.name,
                'file_path': str(img_path),
                'error': str(e)
            })

# Create the single master DataFrame FIRST to ensure data integrity
df_all = pd.DataFrame(records)

In [ ]:
dfs = {}

if not df_all.empty:
    # Safely split the master DataFrame into the requested categories
    dfs['Ant'] = df_all[df_all['region'] == 'Ant'].copy().reset_index(drop=True)
    dfs['Post'] = df_all[df_all['region'] == 'Post'].copy().reset_index(drop=True)
    dfs['L-Lat'] = df_all[(df_all['side'] == 'L') & (df_all['region'] == 'Lat')].copy().reset_index(drop=True)
    dfs['L-Med'] = df_all[(df_all['side'] == 'L') & (df_all['region'] == 'Med')].copy().reset_index(drop=True)
    dfs['R-Lat'] = df_all[(df_all['side'] == 'R') & (df_all['region'] == 'Lat')].copy().reset_index(drop=True)
    dfs['R-Med'] = df_all[(df_all['side'] == 'R') & (df_all['region'] == 'Med')].copy().reset_index(drop=True)
    
    # Define what counts as a perfectly categorized image
    known_conditions = (
        (df_all['region'] == 'Ant') | 
        (df_all['region'] == 'Post') | 
        ((df_all['side'] == 'L') & (df_all['region'] == 'Lat')) |
        ((df_all['side'] == 'L') & (df_all['region'] == 'Med')) |
        ((df_all['side'] == 'R') & (df_all['region'] == 'Lat')) |
        ((df_all['side'] == 'R') & (df_all['region'] == 'Med'))
    )
    
    # Catch anything that didn't fit perfectly (like Lat images missing L/R folders)
    dfs['Unknown'] = df_all[~known_conditions].copy().reset_index(drop=True)

else:
    # Fallback if no images are found
    for key in ['Ant', 'Post', 'L-Lat', 'L-Med', 'R-Lat', 'R-Med', 'Unknown']:
        dfs[key] = pd.DataFrame()

# Print the exact counts
print("--- Image Counts by Category ---")
for key, df in dfs.items():
    print(f"{key} images: {len(df)}")

# Check the total to verify it matches your master dataframe perfectly
total_in_dfs = sum(len(df) for df in dfs.values())
print(f"\nTotal categorized: {total_in_dfs}")
print(f"Master DataFrame size: {len(df_all)}")

In [ ]:
# Remove images if:
# 1) filename contains "scale" (case-insensitive), OR
# 2) file_path contains "unknown" (case-insensitive)
if "file_name" not in df_all.columns or "file_path" not in df_all.columns:
    raise KeyError("df_all must contain both 'file_name' and 'file_path' columns.")

before_total = len(df_all)
remove_mask_all = (
    df_all["file_name"].astype(str).str.contains("scale", case=False, na=False)
    |
    df_all["file_path"].astype(str).str.contains("unknown", case=False, na=False)
)
df_all = df_all.loc[~remove_mask_all].reset_index(drop=True)
removed_total = before_total - len(df_all)

print(f"df_all: removed {removed_total} rows ('scale' in file_name OR 'unknown' in file_path) ({before_total} -> {len(df_all)})")

# Apply the same filter to each dataframe in dfs
if "dfs" in locals() and isinstance(dfs, dict):
    for key, frame in dfs.items():
        if isinstance(frame, pd.DataFrame) and "file_name" in frame.columns and "file_path" in frame.columns:
            before_n = len(frame)
            remove_mask = (
                frame["file_name"].astype(str).str.contains("scale", case=False, na=False)
                |
                frame["file_path"].astype(str).str.contains("unknown", case=False, na=False)
            )
            dfs[key] = frame.loc[~remove_mask].reset_index(drop=True)
            removed_n = before_n - len(dfs[key])
            print(f"{key}: removed {removed_n} rows ({before_n} -> {len(dfs[key])})")

### Recommended pattern
`{subject}_{visit}_{knee_side}_{region}.{ext}`

Build the normalized image name based on the DataFrame the row belongs to:

- `dfs['Ant']` -> `knee_side = NA`, `region = Ant`
- `dfs['Post']` -> `knee_side = NA`, `region = Post`
- `dfs['L-Lat']` -> `knee_side = L`, `region = Lat`
- `dfs['L-Med']` -> `knee_side = L`, `region = Med`
- `dfs['R-Lat']` -> `knee_side = R`, `region = Lat`
- `dfs['R-Med']` -> `knee_side = R`, `region = Med`
- `dfs['Unknown']` -> keep parsed fallback values (`unknownside`, `unknownregion`)

So the naming logic is:
1. Extract `subject`, `visit`, and `ext` from metadata/path.
2. Infer `knee_side` and `region` from the DataFrame name.
3. Concatenate as `{subject}_{visit}_{knee_side}_{region}.{ext}`.

Example:
- Raw: `73NY-9may18-Baseline_post StepD-Knee_Right_Med.tiff`
- Normalized: `NY-73_Baseline_R_Med.tiff`

This normalization should be done before EDA summaries, train/validation splitting, and dataframe merges.

In [ ]:
import re
from typing import Optional, Tuple

# Normalize metadata tokens
def _clean_token(value: Optional[str]) -> str:
    if value is None:
        return "unknown"
    text = str(value).strip().lower()
    text = re.sub(r"\(copy\)|\(\d+\)$", "", text)  # remove copy markers like (1), (2)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text if text else "unknown"


def _extract_visit(file_path: str, file_name: str) -> str:
    text = f"{file_path} {file_name}".lower()

    if "baseline" in text:
        return "Baseline"

    fu_match = re.search(r"\bfu\s*([1-9])\b", text) or re.search(r"\bfu([1-9])\b", text)
    if fu_match:
        return f"FU{fu_match.group(1)}"

    if "end of treatment" in text:
        return "EndOfTreatment"
    if "end of trial" in text:
        return "EndOfTrial"

    return "UnknownVisit"


def _extract_side(file_path: str, file_name: str) -> str:
    text = f"{file_path} {file_name}".lower()
    if re.search(r"\bknee[_\-\s]?left\b|\bleft\b|/l/", text):
        return "L"
    if re.search(r"\bknee[_\-\s]?right\b|\bright\b|/r/", text):
        return "R"
    return "unknownside"


def _extract_region(file_path: str, file_name: str) -> str:
    text = f"{file_path} {file_name}".lower()
    if re.search(r"\blet\b|\blat\b|\blateral\b", text):
        return "Lat"
    if re.search(r"\bmed\b|\bmedial\b", text):
        return "Med"
    if re.search(r"\bant\b|\banterior\b", text):
        return "Ant"
    if re.search(r"\bpost\b|\bposterior\b", text):
        return "Post"
    return "unknownregion"


def _infer_side_region_from_df_name(df_name: str, file_path: str, file_name: str) -> Tuple[str, str]:
    mapping = {
        "Ant": ("NA", "Ant"),
        "Post": ("NA", "Post"),
        "L-Lat": ("L", "Lat"),
        "L-Med": ("L", "Med"),
        "R-Lat": ("R", "Lat"),
        "R-Med": ("R", "Med"),
    }

    if df_name in mapping:
        return mapping[df_name]

    # Unknown dataframe: keep parsed fallback values
    return _extract_side(file_path, file_name), _extract_region(file_path, file_name)


def _normalize_filename_row(row: pd.Series, df_name: str) -> str:
    # Keep subject formatting like NY-73 (not lowercase underscore tokens)
    subject_raw = str(row.get("subject", "unknownsubject")).strip()
    subject = subject_raw if subject_raw else "unknownsubject"

    visit = _extract_visit(str(row.get("file_path", "")), str(row.get("file_name", "")))
    side, region = _infer_side_region_from_df_name(
        df_name=df_name,
        file_path=str(row.get("file_path", "")),
        file_name=str(row.get("file_name", "")),
    )

    ext = str(row.get("extension", "")).strip().lower()
    if not ext.startswith("."):
        ext = f".{ext}" if ext else ".jpg"

    normalized = f"{subject}_{visit}_{side}_{region}{ext}"
    normalized = re.sub(r"_+", "_", normalized).replace("_.", ".")
    return normalized


# Build normalized names from DataFrame membership (matches markdown rules)
normalized_pairs = []
for df_name, frame in dfs.items():
    if frame.empty or "file_path" not in frame.columns:
        continue

    frame_local = frame.copy()
    frame_local["normalized_file_name"] = frame_local.apply(
        lambda row: _normalize_filename_row(row, df_name=df_name),
        axis=1,
    )
    dfs[df_name] = frame_local
    normalized_pairs.append(frame_local[["file_path", "normalized_file_name"]])


# Merge normalized names back into df_all from dfs membership
if normalized_pairs:
    mapping_df = pd.concat(normalized_pairs, ignore_index=True).drop_duplicates(subset=["file_path"] )
    df_all = df_all.merge(mapping_df, on="file_path", how="left")
else:
    df_all["normalized_file_name"] = pd.NA


def _fallback_normalized_for_uncategorized(row: pd.Series) -> str:
    # Safety fallback only if a row did not get a name through dfs mapping
    subject_raw = str(row.get("subject", "unknownsubject")).strip()
    subject = subject_raw if subject_raw else "unknownsubject"
    visit = _extract_visit(str(row.get("file_path", "")), str(row.get("file_name", "")))
    side = _extract_side(str(row.get("file_path", "")), str(row.get("file_name", "")))
    region = _extract_region(str(row.get("file_path", "")), str(row.get("file_name", "")))
    ext = str(row.get("extension", "")).strip().lower()
    if not ext.startswith("."):
        ext = f".{ext}" if ext else ".jpg"
    return f"{subject}_{visit}_{side}_{region}{ext}"


missing_mask = df_all["normalized_file_name"].isna()
if missing_mask.any():
    df_all.loc[missing_mask, "normalized_file_name"] = df_all.loc[missing_mask].apply(
        _fallback_normalized_for_uncategorized,
        axis=1,
    )


# Quick validation
dup_count = int(df_all["normalized_file_name"].duplicated().sum())
print(f"Normalized names created for {len(df_all)} rows.")
print(f"Duplicate normalized names: {dup_count}")
display(df_all[["file_name", "normalized_file_name", "subject", "side", "region"]].head(15))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image

# Set the visual style for our plots
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Distribution of images across regions
plt.figure(figsize=(8, 4))
sns.countplot(data=df_all, x='region', order=['Ant', 'Post', 'Med', 'Lat'], palette='viridis')
plt.title('Number of Thermal Images per Region')
plt.ylabel('Count')
plt.show()

# 2. Check Image Dimensions (Width vs Height)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df_all, x='width', bins=30, kde=True, ax=axes[0], color='coral')
axes[0].set_title('Width Distribution')

sns.histplot(data=df_all, x='height', bins=30, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Height Distribution')

sns.histplot(data=df_all, x='aspect_ratio', bins=30, kde=True, ax=axes[2], color='purple')
axes[2].set_title('Aspect Ratio Distribution')

plt.tight_layout()
plt.show()

# 3. Check for any corrupted images caught in the previous step
if 'corrupted_files' in locals() and len(corrupted_files) > 0:
    print(f"Warning: Found {len(corrupted_files)} corrupted files.")
else:
    print("All image files are intact and readable.")

In [ ]:
# Image-size statistics for ALL images in df_all
required_cols = ["width", "height", "aspect_ratio"]
missing = [c for c in required_cols if c not in df_all.columns]
if missing:
    raise KeyError(f"Missing required columns in df_all: {missing}")

# Work on a safe copy
size_df = df_all[required_cols].copy()
size_df["width"] = pd.to_numeric(size_df["width"], errors="coerce")
size_df["height"] = pd.to_numeric(size_df["height"], errors="coerce")
size_df["aspect_ratio"] = pd.to_numeric(size_df["aspect_ratio"], errors="coerce")
size_df["area"] = size_df["width"] * size_df["height"]

print(f"Total images: {len(df_all)}")
print(f"Valid size rows: {len(size_df.dropna(subset=['width', 'height']))}")

# Full descriptive stats with extra percentiles
size_stats = size_df.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
display(size_stats)

# Additional stats (variance, skewness, kurtosis)
extra_stats = pd.DataFrame({
    "variance": size_df.var(numeric_only=True),
    "skewness": size_df.skew(numeric_only=True),
    "kurtosis": size_df.kurtosis(numeric_only=True),
})
display(extra_stats)

# Exact resolution distribution (all unique WxH combinations)
resolution_counts = (
    df_all.assign(resolution=df_all["width"].astype(str) + "x" + df_all["height"].astype(str))
          .groupby("resolution")
          .size()
          .reset_index(name="count")
          .sort_values(["count", "resolution"], ascending=[False, True])
          .reset_index(drop=True)
)

print(f"Unique resolutions: {len(resolution_counts)}")
display(resolution_counts)

## Image Size Normalization

To prepare the thermal dataset for modeling, all images are resized to a single fixed resolution.  
This ensures every sample has the same shape, which is required for batch training and stable model performance.

- **Why normalize size?**
    - Different image dimensions can break training pipelines.
    - Consistent size reduces memory issues and improves reproducibility.
    - It allows fair comparison between regions and subjects.

- **Important note for this project**
    - Images are treated as **RGB (3 channels)**, not grayscale thermal raw data.
    - Resizing changes spatial dimensions only, while preserving the 3-channel format.

- **Outcome**
    - Each image has uniform width and height.
    - The dataset is ready for preprocessing, augmentation, and model input.

In [ ]:
# Simple resize pipeline: all images -> 256x256 RGB (3 channels)
target_size = (256, 256)
resized_root = Path("Images_256")

resized_records = []
resize_errors = []

for src in df_all["file_path"].astype(str):
    src_path = Path(src)
    dst_path = resized_root / src_path.relative_to(base_dir)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with Image.open(src_path) as im:
            im_rgb = im.convert("RGB")  # enforce 3 channels
            im_resized = im_rgb.resize(target_size, Image.Resampling.BILINEAR)
            im_resized.save(dst_path)

        resized_records.append(
            {
                "file_path": str(src_path),
                "resized_file_path": str(dst_path),
                "resized_width": target_size[0],
                "resized_height": target_size[1],
                "resized_mode": "RGB",
            }
        )
    except Exception as e:
        resize_errors.append(
            {"file_path": str(src_path), "error": str(e)}
        )

df_resized = pd.DataFrame(resized_records)
df_resize_errors = pd.DataFrame(resize_errors)

# Attach resized path metadata back to df_all
df_all = df_all.merge(df_resized, on="file_path", how="left")

print(f"Resized successfully: {len(df_resized)}")
print(f"Failed: {len(df_resize_errors)}")
display(df_resized.head())
if not df_resize_errors.empty:
    display(df_resize_errors.head())

In [ ]:
# Validate that all resized images are the same size
print("=" * 60)
print("IMAGE SIZE VALIDATION")
print("=" * 60)

# Check the resized metadata
if "resized_width" in df_all.columns and "resized_height" in df_all.columns:
    unique_sizes = df_all[["resized_width", "resized_height"]].drop_duplicates()
    
    print(f"\nUnique resized dimensions found: {len(unique_sizes)}")
    display(unique_sizes)
    
    if len(unique_sizes) == 1:
        width = unique_sizes["resized_width"].iloc[0]
        height = unique_sizes["resized_height"].iloc[0]
        print(f"\n✓ SUCCESS: All {len(df_all)} images are uniformly resized to {width}x{height}")
    else:
        print(f"\n✗ WARNING: Found {len(unique_sizes)} different sizes!")
        size_counts = df_all.groupby(["resized_width", "resized_height"]).size().reset_index(name="count")
        display(size_counts)
else:
    print("Resized metadata columns not found. Checking original dimensions...")
    unique_sizes = df_all[["width", "height"]].drop_duplicates()
    print(f"Unique original dimensions: {len(unique_sizes)}")
    display(unique_sizes)

# Display one random thermal image from each region
fig, axes = plt.subplots(1, len(dfs), figsize=(20, 5))

for ax, (region, df) in zip(axes, dfs.items()):
    if not df.empty:
        # Sample one random image path from the region
        sample_path = df.sample(1, random_state=42)['file_path'].values[0]
        img = Image.open(sample_path)
        
        ax.imshow(img)
        ax.set_title(f'Region: {region}\n{img.size[0]}x{img.size[1]}', fontsize=12)
        ax.axis('off')
    else:
        ax.set_title(f'Region: {region} (No Data)')
        ax.axis('off')

plt.suptitle("Sample RGB Thermal Images by Region", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Select a random image to analyze its thermal color distribution
sample_path = df_all.sample(1, random_state=10)['file_path'].values[0]
img_array = np.array(Image.open(sample_path).convert('RGB'))

# Set up the plot
plt.figure(figsize=(10, 5))
colors = ('red', 'green', 'blue')

# Calculate and plot the histogram for each RGB channel
for i, color in enumerate(colors):
    hist, bins = np.histogram(img_array[:, :, i], bins=256, range=(0, 256))
    plt.plot(bins[:-1], hist, color=color, label=f'{color.capitalize()} Channel')

plt.title('RGB Pixel Intensity Distribution (Thermal Profile)')
plt.xlabel('Pixel Intensity (0-255)')
plt.ylabel('Frequency of Pixels')
plt.legend()
plt.show()

In [ ]:
# Calculate mean brightness for a sample of images
def get_mean_brightness(filepath):
    try:
        # Convert to grayscale to get a single brightness value
        img = Image.open(filepath).convert('L')
        return np.mean(np.array(img))
    except:
        return np.nan

# Apply to a subset of data (or all, if the dataset isn't massively huge)
# For speed, we'll take a random sample of 200 images to plot
sample_df = df_all.sample(min(200, len(df_all)), random_state=42).copy()
sample_df['mean_brightness'] = sample_df['file_path'].apply(get_mean_brightness)

plt.figure(figsize=(12, 6))
# Sort by subject to see grouped patterns
sns.boxplot(data=sample_df.sort_values('subject'), x='subject', y='mean_brightness')
plt.title('Distribution of Mean Image Brightness by Subject (Sample)')
plt.xticks(rotation=90)
plt.ylabel('Mean Pixel Intensity (Grayscale)')
plt.show()

## PN Cross-Check: Excel vs Images

This check compares patient IDs (`PN`) between:
- `df_merged` (Excel-derived table)
- `df_all` (images-derived table)

It reports:
1. PNs in Excel but missing from images
2. PNs in images but missing from Excel

In [ ]:
import re
import pandas as pd

# Safety checks
if 'df_merged' not in globals() or not isinstance(df_merged, pd.DataFrame):
    raise NameError("df_merged is not available. Run the Excel preprocessing cells first.")
if 'df_all' not in globals() or not isinstance(df_all, pd.DataFrame):
    raise NameError("df_all is not available. Run the image EDA collection cells first.")
if 'PN' not in df_merged.columns:
    raise KeyError("df_merged must contain a 'PN' column.")
if 'subject' not in df_all.columns:
    raise KeyError("df_all must contain a 'subject' column.")


def normalize_pn(value: object) -> str:
    """Normalize IDs like '73-NY' or 'NY-73' to a canonical form 'NY-73'."""
    text = str(value).strip().upper().replace('_', '-').replace(' ', '')
    text = re.sub(r'[^A-Z0-9-]', '', text)

    num_first = re.match(r'^(\d{1,3})-([A-Z]{1,4})$', text)
    if num_first:
        num = num_first.group(1).zfill(2)
        letters = num_first.group(2)
        return f"{letters}-{num}"

    letters_first = re.match(r'^([A-Z]{1,4})-(\d{1,3})$', text)
    if letters_first:
        letters = letters_first.group(1)
        num = letters_first.group(2).zfill(2)
        return f"{letters}-{num}"

    return text


excel_pn_set = {
    normalize_pn(v)
    for v in df_merged['PN'].dropna().astype(str)
    if str(v).strip()
}

image_pn_set = {
    normalize_pn(v)
    for v in df_all['subject'].dropna().astype(str)
    if str(v).strip()
}

excel_not_in_images = sorted(excel_pn_set - image_pn_set)
images_not_in_excel = sorted(image_pn_set - excel_pn_set)

print('=== PN Cross-Check Summary ===')
print(f"Unique PN in Excel (df_merged): {len(excel_pn_set)}")
print(f"Unique PN in Images (df_all.subject): {len(image_pn_set)}")
print(f"In Excel but not in Images: {len(excel_not_in_images)}")
print(f"In Images but not in Excel: {len(images_not_in_excel)}")

excel_missing_df = pd.DataFrame({'PN_in_excel_not_in_images': excel_not_in_images})
images_missing_df = pd.DataFrame({'PN_in_images_not_in_excel': images_not_in_excel})

print('\nPNs in Excel but missing in Images:')
display(excel_missing_df)

print('\nPNs in Images but missing in Excel:')
display(images_missing_df)

In [ ]:
# Keep only PNs that exist in BOTH sources (Excel and Images)
import re
import pandas as pd

if 'df_merged' not in globals() or 'df_all' not in globals():
    raise NameError("df_merged and df_all must exist before running this cell.")
if 'PN' not in df_merged.columns:
    raise KeyError("df_merged must contain a 'PN' column.")
if 'subject' not in df_all.columns:
    raise KeyError("df_all must contain a 'subject' column.")


def normalize_pn(value: object) -> str:
    text = str(value).strip().upper().replace('_', '-').replace(' ', '')
    text = re.sub(r'[^A-Z0-9-]', '', text)

    num_first = re.match(r'^(\d{1,3})-([A-Z]{1,4})$', text)
    if num_first:
        return f"{num_first.group(2)}-{num_first.group(1).zfill(2)}"

    letters_first = re.match(r'^([A-Z]{1,4})-(\d{1,3})$', text)
    if letters_first:
        return f"{letters_first.group(1)}-{letters_first.group(2).zfill(2)}"

    return text


# Build normalized PN columns for filtering
excel_before_rows = len(df_merged)
images_before_rows = len(df_all)

df_merged['_PN_norm'] = df_merged['PN'].apply(normalize_pn)
df_all['_PN_norm'] = df_all['subject'].apply(normalize_pn)

excel_set_before = set(df_merged['_PN_norm'].dropna())
images_set_before = set(df_all['_PN_norm'].dropna())

only_excel_before = sorted(excel_set_before - images_set_before)
only_images_before = sorted(images_set_before - excel_set_before)
shared_pn = excel_set_before & images_set_before

print('=== Before Filtering (from current data) ===')
print(f"Unique PN in Excel: {len(excel_set_before)}")
print(f"Unique PN in Images: {len(images_set_before)}")
print(f"Excel-only PN: {len(only_excel_before)}")
print(f"Images-only PN: {len(only_images_before)}")

# Filter both dataframes to the intersection only
df_merged = df_merged[df_merged['_PN_norm'].isin(shared_pn)].copy()
df_all = df_all[df_all['_PN_norm'].isin(shared_pn)].copy()

excel_after_rows = len(df_merged)
images_after_rows = len(df_all)

excel_set_after = set(df_merged['_PN_norm'].dropna())
images_set_after = set(df_all['_PN_norm'].dropna())

only_excel_after = sorted(excel_set_after - images_set_after)
only_images_after = sorted(images_set_after - excel_set_after)

print('\n=== After Filtering to Shared PN Only ===')
print(f"Rows in df_merged: {excel_before_rows} -> {excel_after_rows}")
print(f"Rows in df_all: {images_before_rows} -> {images_after_rows}")
print(f"Unique PN in Excel (after): {len(excel_set_after)}")
print(f"Unique PN in Images (after): {len(images_set_after)}")
print(f"Excel-only PN after filter: {len(only_excel_after)}")
print(f"Images-only PN after filter: {len(only_images_after)}")

# Explicit check against your previous mismatch output (12 and 2)
print('\n=== Check Against Previous Output You Shared ===')
print(f"Expected before filter -> Excel-only: 12, Images-only: 2")
print(f"Observed before filter -> Excel-only: {len(only_excel_before)}, Images-only: {len(only_images_before)}")

# Cleanup helper columns
df_merged.drop(columns=['_PN_norm'], inplace=True, errors='ignore')
df_all.drop(columns=['_PN_norm'], inplace=True, errors='ignore')

## PN + Visit Cross-Check: Excel vs Images

This check compares **PN + Visit** combinations between:
- `df_merged` (Excel-derived table)
- `df_all` (images-derived table)

Visit mapping used for comparison:
- `Visit == 1` -> `Baseline`
- `Visit == 2` -> `FU1`
- `Visit == 3` -> `FU3`

It reports:
1. PN+Visit pairs in Excel but missing from images
2. PN+Visit pairs in images but missing from Excel

In [ ]:
import re
import pandas as pd

# Safety checks
if 'df_merged' not in globals() or not isinstance(df_merged, pd.DataFrame):
    raise NameError("df_merged is not available. Run the Excel preprocessing cells first.")
if 'df_all' not in globals() or not isinstance(df_all, pd.DataFrame):
    raise NameError("df_all is not available. Run the image EDA cells first.")
if 'PN' not in df_merged.columns:
    raise KeyError("df_merged must contain a 'PN' column.")
if 'Visit' not in df_merged.columns:
    raise KeyError("df_merged must contain a 'Visit' column.")
if 'subject' not in df_all.columns:
    raise KeyError("df_all must contain a 'subject' column.")


def normalize_pn(value: object) -> str:
    """Normalize IDs like '73-NY' or 'NY-73' to canonical 'NY-73'."""
    text = str(value).strip().upper().replace('_', '-').replace(' ', '')
    text = re.sub(r'[^A-Z0-9-]', '', text)

    num_first = re.match(r'^(\d{1,3})-([A-Z]{1,4})$', text)
    if num_first:
        return f"{num_first.group(2)}-{num_first.group(1).zfill(2)}"

    letters_first = re.match(r'^([A-Z]{1,4})-(\d{1,3})$', text)
    if letters_first:
        return f"{letters_first.group(1)}-{letters_first.group(2).zfill(2)}"

    return text


def extract_visit_label_from_image_row(row: pd.Series) -> str:
    """Infer visit label from image metadata/path using requested mapping labels."""
    text = f"{row.get('file_path', '')} {row.get('file_name', '')} {row.get('normalized_file_name', '')}".lower()

    if 'baseline' in text:
        return 'Baseline'
    if re.search(r'\bfu\s*1\b|\bfu1\b', text):
        return 'FU1'
    if re.search(r'\bfu\s*3\b|\bfu3\b', text):
        return 'FU3'

    return 'UnknownVisit'


# Requested mapping from numeric Visit to labels
visit_map = {
    1: 'Baseline',
    2: 'FU1',
    3: 'FU3',
}

# Build Excel PN+Visit set
excel_tmp = df_merged[['PN', 'Visit']].copy()
excel_tmp['PN_norm'] = excel_tmp['PN'].apply(normalize_pn)
excel_tmp['Visit_label'] = pd.to_numeric(excel_tmp['Visit'], errors='coerce').map(visit_map)
excel_tmp = excel_tmp.dropna(subset=['PN_norm', 'Visit_label'])

excel_pairs = {
    (pn, visit)
    for pn, visit in excel_tmp[['PN_norm', 'Visit_label']].drop_duplicates().itertuples(index=False)
}

# Build Image PN+Visit set
img_tmp = df_all.copy()
img_tmp['PN_norm'] = img_tmp['subject'].apply(normalize_pn)
img_tmp['Visit_label'] = img_tmp.apply(extract_visit_label_from_image_row, axis=1)
unknown_visit_count = int((img_tmp['Visit_label'] == 'UnknownVisit').sum())
img_tmp = img_tmp[img_tmp['Visit_label'].isin({'Baseline', 'FU1', 'FU3'})]

image_pairs = {
    (pn, visit)
    for pn, visit in img_tmp[['PN_norm', 'Visit_label']].drop_duplicates().itertuples(index=False)
}

# Compare sets
excel_not_in_images = sorted(excel_pairs - image_pairs)
images_not_in_excel = sorted(image_pairs - excel_pairs)

print('=== PN + Visit Cross-Check Summary ===')
print(f"Unique PN+Visit pairs in Excel: {len(excel_pairs)}")
print(f"Unique PN+Visit pairs in Images: {len(image_pairs)}")
print(f"In Excel but not in Images: {len(excel_not_in_images)}")
print(f"In Images but not in Excel: {len(images_not_in_excel)}")

excel_missing_df = pd.DataFrame(excel_not_in_images, columns=['PN', 'Visit'])
images_missing_df = pd.DataFrame(images_not_in_excel, columns=['PN', 'Visit'])

print('\nPN+Visit in Excel but missing in Images:')
display(excel_missing_df)

print('\nPN+Visit in Images but missing in Excel:')
display(images_missing_df)

print(f"\nRows with unmapped image visit label (ignored in comparison): {unknown_visit_count}")

In [ ]:
import re
import pandas as pd

# Safety checks
if "df_all" not in globals() or not isinstance(df_all, pd.DataFrame):
    raise NameError("df_all is not available. Run the image EDA cells first.")
if "subject" not in df_all.columns:
    raise KeyError("df_all must contain a 'subject' column.")
if "file_path" not in df_all.columns or "file_name" not in df_all.columns:
    raise KeyError("df_all must contain 'file_path' and 'file_name' columns.")

# Reuse your existing function if available, otherwise define it
if "extract_visit_label_from_image_row" not in globals():
    def extract_visit_label_from_image_row(row: pd.Series) -> str:
        text = f"{row.get('file_path', '')} {row.get('file_name', '')} {row.get('normalized_file_name', '')}".lower()
        if "baseline" in text:
            return "Baseline"
        if re.search(r"\bfu\s*1\b|\bfu1\b", text):
            return "FU1"
        if re.search(r"\bfu\s*3\b|\bfu3\b", text):
            return "FU3"
        return "UnknownVisit"

inspect_df = df_all.copy()
if "normalized_file_name" not in inspect_df.columns:
    inspect_df["normalized_file_name"] = ""

inspect_df["Visit_label"] = inspect_df.apply(extract_visit_label_from_image_row, axis=1)

mapped_df = inspect_df[inspect_df["Visit_label"].isin(["Baseline", "FU1", "FU3"])].copy()
unknown_df = inspect_df[inspect_df["Visit_label"] == "UnknownVisit"].copy()

print("=== Visit Label Detection Summary ===")
print(inspect_df["Visit_label"].value_counts(dropna=False).sort_index())
print(f"\nMapped rows (Baseline/FU1/FU3): {len(mapped_df)}")
print(f"UnknownVisit rows: {len(unknown_df)}")

cols_to_show = ["subject", "file_name", "file_path", "normalized_file_name", "Visit_label"]
cols_to_show = [c for c in cols_to_show if c in inspect_df.columns]

print("\n=== Examples that DO match the use-case (Baseline/FU1/FU3) ===")
display(mapped_df[cols_to_show].sample(min(15, len(mapped_df)), random_state=42))

print("\n=== Examples that are UnknownVisit (the ignored rows) ===")
display(unknown_df[cols_to_show].head(33))

In [ ]:
import re
import pandas as pd

# Safety checks
if "df_all" not in globals() or not isinstance(df_all, pd.DataFrame):
    raise NameError("df_all is not available. Run the image EDA cells first.")
if "dfs" not in globals() or not isinstance(dfs, dict):
    raise NameError("dfs is not available. Run the image categorization cells first.")
if "file_path" not in df_all.columns or "file_name" not in df_all.columns:
    raise KeyError("df_all must contain 'file_path' and 'file_name' columns.")

# Reuse existing function if present; otherwise define it
if "extract_visit_label_from_image_row" not in globals():
    def extract_visit_label_from_image_row(row: pd.Series) -> str:
        text = f"{row.get('file_path', '')} {row.get('file_name', '')} {row.get('normalized_file_name', '')}".lower()
        if "baseline" in text:
            return "Baseline"
        if re.search(r"\bfu\s*1\b|\bfu1\b", text):
            return "FU1"
        if re.search(r"\bfu\s*3\b|\bfu3\b", text):
            return "FU3"
        return "UnknownVisit"

inspect_df = df_all.copy()
if "normalized_file_name" not in inspect_df.columns:
    inspect_df["normalized_file_name"] = ""

inspect_df["Visit_label"] = inspect_df.apply(extract_visit_label_from_image_row, axis=1)
unknown_df = inspect_df[inspect_df["Visit_label"] == "UnknownVisit"].copy()

# Build file_path -> dataframe name(s) mapping from dfs
path_to_df_names = {}
for df_name, frame in dfs.items():
    if isinstance(frame, pd.DataFrame) and "file_path" in frame.columns:
        for p in frame["file_path"].dropna().astype(str).unique():
            path_to_df_names.setdefault(p, []).append(df_name)

unknown_df["source_image_df"] = unknown_df["file_path"].astype(str).map(
    lambda p: ", ".join(sorted(path_to_df_names.get(p, []))) if path_to_df_names.get(p) else "Not found in dfs"
)

print("=== UnknownVisit rows by source image dataframe ===")
print(f"Total UnknownVisit rows: {len(unknown_df)}")
print("\nCounts by source_image_df:")
display(
    unknown_df["source_image_df"]
    .value_counts(dropna=False)
    .rename_axis("source_image_df")
    .reset_index(name="count")
)

cols_to_show = [
    "subject", "file_name", "file_path", "normalized_file_name", "Visit_label", "source_image_df"
]
cols_to_show = [c for c in cols_to_show if c in unknown_df.columns]

print("\nAll UnknownVisit rows with source image dataframe:")
display(unknown_df[cols_to_show].reset_index(drop=True))

## Image Labeling by PN + Visit

This step creates image-level labels by connecting each image in `df_all` to clinical rows in `df_merged`.

Matching logic:
1. Normalize patient ID (`PN` / `subject`) to the same format.
2. Extract visit from image metadata/name and map it to visit number:
   - `Baseline -> 1`
   - `FU1 -> 2`
   - `FU3 -> 3`
3. Match on `(PN_norm, Visit)` between images and clinical table.
4. Assign `pain_label` to each image from `df_merged` (uses existing `pain_label` if available, otherwise `PF_DIAGNOSIS`).

In [ ]:
import re
import pandas as pd

# Safety checks
if 'df_all' not in globals() or not isinstance(df_all, pd.DataFrame):
    raise NameError("df_all is not available. Run the image EDA cells first.")
if 'df_merged' not in globals() or not isinstance(df_merged, pd.DataFrame):
    raise NameError("df_merged is not available. Run the Excel preprocessing cells first.")
if 'subject' not in df_all.columns:
    raise KeyError("df_all must contain 'subject'.")
if 'PN' not in df_merged.columns or 'Visit' not in df_merged.columns:
    raise KeyError("df_merged must contain both 'PN' and 'Visit'.")
if 'file_path' not in df_all.columns or 'file_name' not in df_all.columns:
    raise KeyError("df_all must contain both 'file_path' and 'file_name'.")


def normalize_pn(value: object) -> str:
    text = str(value).strip().upper().replace('_', '-').replace(' ', '')
    text = re.sub(r'[^A-Z0-9-]', '', text)

    num_first = re.match(r'^(\d{1,3})-([A-Z]{1,4})$', text)
    if num_first:
        return f"{num_first.group(2)}-{num_first.group(1).zfill(2)}"

    letters_first = re.match(r'^([A-Z]{1,4})-(\d{1,3})$', text)
    if letters_first:
        return f"{letters_first.group(1)}-{letters_first.group(2).zfill(2)}"

    return text


def extract_visit_label_from_image_row(row: pd.Series) -> str:
    text = f"{row.get('file_path', '')} {row.get('file_name', '')} {row.get('normalized_file_name', '')}".lower()
    if 'baseline' in text:
        return 'Baseline'
    if re.search(r'\bfu\s*1\b|\bfu1\b', text):
        return 'FU1'
    if re.search(r'\bfu\s*3\b|\bfu3\b', text):
        return 'FU3'
    return 'UnknownVisit'


visit_label_to_num = {'Baseline': 1, 'FU1': 2, 'FU3': 3}

# Build image keys
img = df_all.copy()
if 'normalized_file_name' not in img.columns:
    img['normalized_file_name'] = ''

img['PN_norm'] = img['subject'].apply(normalize_pn)
img['Visit_label'] = img.apply(extract_visit_label_from_image_row, axis=1)
img['Visit'] = img['Visit_label'].map(visit_label_to_num)

# Keep only images with recognized visits
img_known = img[img['Visit'].notna()].copy()
img_known['Visit'] = img_known['Visit'].astype(int)

# Choose label source in df_merged
label_source_col = 'pain_label' if 'pain_label' in df_merged.columns else 'PF_DIAGNOSIS'
if label_source_col not in df_merged.columns:
    raise KeyError("df_merged must contain 'pain_label' or 'PF_DIAGNOSIS' for labeling.")

clin = df_merged[['PN', 'Visit', label_source_col]].copy()
clin['PN_norm'] = clin['PN'].apply(normalize_pn)
clin['Visit'] = pd.to_numeric(clin['Visit'], errors='coerce')
clin = clin.dropna(subset=['PN_norm', 'Visit'])
clin['Visit'] = clin['Visit'].astype(int)
clin['pain_label'] = pd.to_numeric(clin[label_source_col], errors='coerce')
clin = clin[['PN_norm', 'Visit', 'pain_label']].drop_duplicates()

# Merge image rows with clinical labels by PN + Visit
df_images_labeled = img_known.merge(
    clin,
    on=['PN_norm', 'Visit'],
    how='left'
)

matched = int(df_images_labeled['pain_label'].notna().sum())
unmatched = int(df_images_labeled['pain_label'].isna().sum())

print('=== Image Labeling Summary ===')
print(f"Label source from df_merged: {label_source_col}")
print(f"Images with recognized visit label: {len(df_images_labeled)}")
print(f"Matched image labels: {matched}")
print(f"Unmatched image labels: {unmatched}")

print('\nPain label distribution (including NaN):')
print(df_images_labeled['pain_label'].value_counts(dropna=False).sort_index())

display_cols = [
    'subject', 'file_name', 'Visit_label', 'Visit', 'PN_norm', 'pain_label', 'file_path'
]
display_cols = [c for c in display_cols if c in df_images_labeled.columns]

print('\nSample labeled images:')
display(df_images_labeled[display_cols].head(20))

if unmatched > 0:
    print('\nSample unmatched rows (for debugging):')
    display(df_images_labeled[df_images_labeled['pain_label'].isna()][display_cols].head(20))

In [ ]:
# Simple X and t splitting: X = images, t = pain_label

# Filter to only rows with valid labels
df_train = df_images_labeled[df_images_labeled['pain_label'].notna()].copy()

# X: file paths to resized images
X = df_train['resized_file_path'].values

# t: pain labels
t = df_train['pain_label'].values

print(f"Dataset size: {len(df_train)}")
print(f"X shape: {X.shape}")
print(f"t shape: {t.shape}")
print(f"\nPain label distribution:")
print(pd.Series(t).value_counts(dropna=False).sort_index())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Show 2 sample images from each class in df_images_labeled
if 'df_images_labeled' not in globals():
    raise NameError("df_images_labeled is not available. Run the image labeling cell first.")

plot_df = df_images_labeled.copy()
if 'pain_label' not in plot_df.columns:
    raise KeyError("df_images_labeled must contain a 'pain_label' column.")

plot_df = plot_df[plot_df['pain_label'].notna()].copy()
plot_df['pain_label'] = pd.to_numeric(plot_df['pain_label'], errors='coerce')
plot_df = plot_df[plot_df['pain_label'].notna()].copy()

if plot_df.empty:
    raise ValueError("No labeled images were found to plot.")

classes = sorted(plot_df['pain_label'].unique())
nrows = len(classes)
ncols = 2
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8, 4 * nrows))
axes = np.atleast_2d(axes)

for row_idx, label in enumerate(classes):
    class_df = plot_df[plot_df['pain_label'] == label].sample(
        n=min(2, len(plot_df[plot_df['pain_label'] == label])),
        random_state=42
    )

    for col_idx in range(ncols):
        ax = axes[row_idx, col_idx]

        if col_idx < len(class_df):
            row = class_df.iloc[col_idx]
            img_path = row['resized_file_path'] if 'resized_file_path' in row and pd.notna(row['resized_file_path']) else row['file_path']
            with Image.open(img_path) as img:
                ax.imshow(img)
            ax.set_title(f"Class {label} - {row.get('Visit_label', 'Visit')}")
        else:
            ax.axis('off')
            continue

        ax.axis('off')

plt.suptitle('Two Sample Images from Each Class', fontsize=16)
plt.tight_layout()
plt.show()